### 0) Imports and warning supression

In [ ]:
from pathlib import Path
import sys
import warnings
warnings.simplefilter('ignore', FutureWarning)
warnings.filterwarnings('ignore', message='IProgress not found')

import numpy as np
import torch
import json
import pandas as pd

# Ensure the repo root is on sys.path so 'release' is importable
repo_root = Path('.').resolve().parent  # assumes notebook is in release/
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from release.inference_api import run_inference_latent, run_inference_phys
from release import config as cfg

# Point to the bundled pre-trained model
run_dir = Path('chosen_model')
print(f'run_dir = {run_dir.resolve()}')

### 1) Load precompted taus for scaling

In [ ]:
taus_path = run_dir / "taus.json"
taus = None
if taus_path.exists():
    with taus_path.open("r") as f:
        taus = json.load(f)
taus

### 2) Run Inference

In [ ]:
import torch as _torch
meta = _torch.load(run_dir / "data_meta.pt", map_location="cpu", weights_only=False)

Din = meta["x_test_raw"].shape[1]
print("Din =", Din)
print("Targets =", cfg.TARGETS)

N = 16
x = torch.rand(N, Din)  # dummy raw inputs

in_cols = list(cfg.FEATURES) + [f"{t}_HS" for t in cfg.TARGETS]

print("\nExpected input column order (x_raw):")
print("Din =", len(in_cols))
for i, c in enumerate(in_cols):
    print(f"  [{i:02d}] {c}")

### 2.1) Latent variable outputs

In [ ]:
mu_lat, std_ale_lat, std_epi_lat, mu_lat_s, std_ale_lat_s = run_inference_latent(
    run_dir, x, num_mc=50, seed=0
)

print("mu_lat:", mu_lat.shape)
print("std_ale_lat:", None if std_ale_lat is None else std_ale_lat.shape)
print("std_epi_lat:", std_epi_lat.shape)

mu_lat[:2]

### 2.2 Physical space outputs

In [ ]:
mean_phys, std_ale_phys, std_epi_phys, std_tot_phys, q_out = run_inference_phys(
    run_dir, x, num_mc=50, seed=0, taus=taus, L=50, quantiles=(0.05, 0.5, 0.95)
)

print("mean_phys:", mean_phys.shape)
print("std_tot_phys:", std_tot_phys.shape)
print("q_out keys:", list(q_out.keys()))
print("Targets:", cfg.TARGETS)

mean_phys[:2], std_tot_phys[:2]

In [ ]:
targets = cfg.TARGETS

# build a output table
out = {}
for j, t in enumerate(targets):
    out[f"mean_{t}"] = mean_phys[:, j]
    out[f"std_tot_{t}"] = std_tot_phys[:, j]
    out[f"std_ale_{t}"] = std_ale_phys[:, j]
    out[f"std_epi_{t}"] = std_epi_phys[:, j]
    out[f"q05_{t}"] = q_out["q05"][:, j]
    out[f"q50_{t}"] = q_out["q50"][:, j]
    out[f"q95_{t}"] = q_out["q95"][:, j]

df_results = pd.DataFrame(out)

# show a few rows
df_results.head().style.format("{:.6f}")

### 2.2.1 Sanity check of error propogation

In [ ]:
max_err = np.max(np.abs(std_tot_phys**2 - (std_ale_phys**2 + std_epi_phys**2)))
print(max_err)